### Create Data

In [0]:
data = [
    (1, "A", 5000),
    (2, "B", 6000),
    (3, "C", 7000)
]

columns = ["id", "name", "salary"]

df = spark.createDataFrame(data, columns)
df.display()

id,name,salary
1,A,5000
2,B,6000
3,C,7000


### Write as Delta Table

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .save("/Volumes/workspace/default/day6")

### Read Delta Table

In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/default/day6")
df.display()

id,name,salary
1,A,5000
2,B,6000
3,C,7000


### INSERT

In [0]:
new_data = [(4, "D", 8000)]
new_df = spark.createDataFrame(new_data, columns)

new_df.write.format("delta")\
    .mode("append")\
    .save("/Volumes/workspace/default/day6")

### UPDATE

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "/Volumes/workspace/default/day6")

delta_table.update(
    condition = "id = 2",
    set = {"salary": "6500"}
)

num_affected_rows
1


### DELETE

In [0]:
delta_table.delete("id = 1")

DataFrame[num_affected_rows: bigint]

### MERGE INTO

### How do we handle incremental load with updates?”

### Existing Data

In [0]:
target_df = spark.read.format("delta").load("/Volumes/workspace/default/day6")
target_df.display()

id,name,salary
2,B,6500
3,C,7000
4,D,8000


### New Data

In [0]:
updates = [
    (2, "B", 7000),  # update
    (5, "E", 9000)   # insert
]

updates_df = spark.createDataFrame(updates, columns)
updates_df.display()

id,name,salary
2,B,7000
5,E,9000


### MERGE

In [0]:
delta_table.alias("target").merge(
    updates_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set={
    "salary": "source.salary"
}).whenNotMatchedInsert(values={
    "id": "source.id",
    "name": "source.name",
    "salary": "source.salary"
}).execute()

### If id exists → UPDATE
### If id not exists → INSERT
### This is UPSERT

### Schema Evolution

### Problem:

### New column arrives

### Example:

In [0]:
new_data = [(6, "F", 10000, "India")]
new_columns = ["id", "name", "salary", "country"]

df_new = spark.createDataFrame(new_data, new_columns)
df_new.display()

id,name,salary,country
6,F,10000,India


### Enable schema evolution:

In [0]:
df_new.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("/Volumes/workspace/default/day6")

In [0]:
df_read=spark.read.format('delta').load("/Volumes/workspace/default/day6")
df_read.display()


id,name,salary,country
6,F,10000,India
2,B,6500,null
3,C,7000,null
4,D,8000,null


### Time Travel

### View History

In [0]:
delta_table.history().display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
8,2026-04-13T17:15:49.000Z,71250968532154,sravanazure4@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(4239667437489104),6455defa-bdcf-4890-b2a9-c8fc09f9a5b0,0413-153639-7mglcn0w-v2n,7,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1199)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
7,2026-04-13T17:07:04.000Z,71250968532154,sravanazure4@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4239667437489104),79d685f8-a491-4e94-a52d-2804fe885ba2,0413-153639-7mglcn0w-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1061, p25FileSize -> 1045, numDeletionVectorsRemoved -> 1, minFileSize -> 1045, numAddedFiles -> 1, maxFileSize -> 1045, p75FileSize -> 1045, p50FileSize -> 1045, numAddedBytes -> 1045)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
6,2026-04-13T17:07:02.000Z,71250968532154,sravanazure4@gmail.com,DELETE,"Map(predicate -> [""(id#11987L = 1)""])",null,List(4239667437489104),79d685f8-a491-4e94-a52d-2804fe885ba2,0413-153639-7mglcn0w-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1905, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1310, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 594)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
5,2026-04-13T17:02:00.000Z,71250968532154,sravanazure4@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4239667437489104),822c4c3b-1646-41d2-8db1-cfea99901c73,0413-153639-7mglcn0w-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 2065, p25FileSize -> 1061, numDeletionVectorsRemoved -> 1, minFileSize -> 1061, numAddedFiles -> 1, maxFileSize -> 1061, p75FileSize -> 1061, p50FileSize -> 1061, numAddedBytes -> 1061)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
4,2026-04-13T17:01:57.000Z,71250968532154,sravanazure4@gmail.com,UPDATE,"Map(predicate -> [""(id#11666L = 2)""])",null,List(4239667437489104),822c4c3b-1646-41d2-8db1-cfea99901c73,0413-153639-7mglcn0w-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2740, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1329, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1004, rewriteTimeMs -> 1410)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
3,2026-04-13T17:01:48.000Z,71250968532154,sravanazure4@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4239667437489104),4ff69b3f-0b46-44be-bfba-ce4754e052d6,0413-153639-7mglcn0w-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 3051, p25FileSize -> 1061, numDeletionVectorsRemoved -> 1, minFileSize -> 1061, numAddedFiles -> 1, maxFileSize -> 1061, p75FileSize -> 1061, p50FileSize -> 1061, numAddedBytes -> 1061)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
2,2026-04-13T17:01:44.000Z,71250968532154,sravanazure4@gmail.com,UPDATE,"Map(predicate -> [""(id#11322L = 2)""])",null,List(4239667437489104),4ff69b3f-0b46-44be-bfba-ce4754e052d6,0413-153639-7mglcn0w-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 5132, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2768, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1004, rewriteT

### Read Old Version

In [0]:
df_old = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/Volumes/workspace/default/day6")
df_old.display()

id,name,salary
1,A,5000
2,B,6000
3,C,7000


### Restore Table

In [0]:
delta_table.restoreToVersion(0)

DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]